# 00_03 Serverless Compute in Free Edition

## Goal
Understand where your code runs in Databricks Free Edition, what serverless means, and how it differs from classic compute.

## Why it matters
Every performance, cost and configuration question in Databricks starts with the compute. Interviewers often ask about compute types, and you need to know what your Free Edition can and cannot show you.

## Concept
**Compute** is the set of machines that run your code. Databricks has two broad styles:

| | Classic compute | Serverless compute |
|---|---|---|
| Who manages machines | You (choose size, workers, runtime) | Databricks |
| Startup time | Minutes | Seconds |
| Configuration | Many settings (node types, autoscaling, init scripts, Spark configs) | Minimal, many settings are fixed |
| Billing idea | Pay for VMs while running | Pay for usage |

Free Edition gives you **serverless only**, with usage quotas. That is enough to learn Spark, Delta and Unity Catalog, but you cannot build custom clusters here.

## Real-world example
A team runs nightly ETL on job compute sized for the workload, while analysts use serverless SQL warehouses for quick queries. Choosing the right one is a cost and performance decision.

## Compute types (know these for interviews)

| Type | Used for |
|---|---|
| All-purpose compute | Interactive development, shared by users |
| Job compute | Automated jobs, created for a run and terminated after |
| SQL warehouse | SQL queries and dashboards |
| Serverless | Managed compute with no cluster configuration |
| Instance pools | Pre-warmed VMs to reduce classic cluster startup (paid workspaces) |

In Free Edition you will mainly use serverless notebooks and a serverless SQL warehouse.

# 3. Compute basics

## 3.1 What am I running on?

**What it does:** prints the Spark version, Python version and the CPU count of the driver's Python process.
**Why it matters:** gives a baseline for the environment. Numbers vary by environment, so note what you see.
**Expected output:** three lines with version and CPU information.

In [0]:
import os, platform

print("Spark.version : ", spark.version )
print("Python version : " , platform.python_version())
print("Drive CPU count (Python Side) : ", os.cpu_count())

## 3.2 Reading a Spark setting safely

**What it does:** tries to read `spark.sql.shuffle.partitions` and handles the case where serverless doesn't allow it.
**Why it matters:** serverless restricts many Spark configs. The try/except pattern lets you probe a setting without breaking the notebook.
**Expected output:** a value (often `auto`) or a message that it can't be read.

In [0]:
try:
    print("shuffle partitons : ", spark.conf.get("spark.sql.shuffle.partitions"))
except Exception as e:
    print("Setting not readable here : ", e)

## 3.3 A small timing test

**What it does:** counts 10 million generated rows and measures how long it takes.
**Why it matters:** gives a rough feel for serverless speed. Running it twice shows the effect of warm-up.
**Expected output:** `rows: 10000000` and a time of a few seconds or less.

In [0]:
import os, platform, time

print("Spark version:", spark.version)
print("Python version:", platform.python_version())
try:
    print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
except Exception:
    print("shuffle partitions: not readable on serverless")

times = []
for _ in range(2):
    start = time.time()
    spark.range(10_000_000).count()
    times.append(round(time.time() - start, 2))
print("10M row count (run 1 / run 2):", times[0], "s /", times[1], "s")

## My observations

Spark version: 4.2.0
Python version: 3.12.3
shuffle partitions: auto
10M row count (run 1 / run 2): 0.29 s / 0.37 s

- What I noticed: the first run includes warm-up, so the second run is usually faster. Serverless starts quickly and needs no cluster setup, but it exposes fewer settings than classic compute.

## Free Edition limits to remember (verified in the UI)

- Serverless compute only
- Usage quotas apply
- No custom clusters, init scripts or instance pools
- Some enterprise features are unavailable

Date I verified this: 2026-10-05

## Interview Q&A

1. **What is serverless compute?**
   Compute where Databricks manages the infrastructure: no cluster sizing, fast startup, and billing based on usage.

2. **All-purpose vs job compute?**
   All-purpose is for interactive work and can be shared. Job compute is created for an automated run and terminates afterward, which is cheaper for production.

3. **Why use job compute for production instead of an all-purpose cluster?**
   Lower cost, isolation from interactive workloads, and a clean environment for each run.

4. **What are instance pools?**
   Pools of idle, pre-warmed VMs that reduce classic cluster startup time.

5. **What is the trade-off of serverless?**
   Less control over configuration in exchange for simplicity and fast startup.